# Leuven.cool Citizen Science Urban Weather Station Network

**Categoría:** Meteorología · **Tamaño:** 12.9 GB · **Formato:** CSV, CSV.gz
**Licencia:** CC-BY-NC-4.0 (No comercial) · [Registro en Zenodo](https://zenodo.org/records/14893734) · [Ficha en el CSDH](https://data.ibercivis.es/datasets/leuven-urban-weather/)

Más de 1.000 millones de observaciones crudas de ~110 estaciones meteorológicas de bajo coste distribuidas por Leuven (Bélgica), con resolución de 16 segundos de junio 2019 a marzo 2025.

Los datos están montados en modo **solo lectura** en `/srv/data/leuven-urban-weather/`.
Todo lo que guardes, hazlo en tu carpeta personal (`~/`).


## Qué hay en el dataset

In [ ]:
from pathlib import Path

DATA = Path('/srv/data/leuven-urban-weather')

for f in sorted(DATA.rglob('*')):
    if f.is_file():
        print(f"{f.relative_to(DATA)}  ({f.stat().st_size/1e6:,.1f} MB)")


## Cargar los datos

El dataset viene en CSV. En el mundo real los CSV no siempre son uniformes (separador `,` o `;`, codificación UTF-8 o Latin-1), así que usamos un cargador que lo detecta. Limitamos a 100.000 filas para explorar con agilidad — quita `nrows` cuando quieras trabajar con todo.

In [ ]:
import pandas as pd

csvs = sorted(DATA.rglob('*.csv')) + sorted(DATA.rglob('*.csv.gz')) + sorted(DATA.rglob('*.gz'))
print('Usando:', csvs[0].name)

def carga_csv(path, **kw):
    """Lector robusto: detecta el separador y prueba utf-8 y latin-1."""
    for enc in ('utf-8', 'latin-1'):
        try:
            return pd.read_csv(path, sep=None, engine='python', encoding=enc, **kw)
        except UnicodeDecodeError:
            continue

df = carga_csv(csvs[0], nrows=100_000)
df.head()


## Primer vistazo

Dimensiones, tipos y estadísticos básicos.

In [ ]:
df.info()
df.describe(include='all').T.head(20)


## Una primera gráfica

Histograma de la primera columna numérica — sustitúyela por la variable que te interese.

In [ ]:
import matplotlib.pyplot as plt

num = df.select_dtypes('number')
if num.shape[1]:
    col = num.columns[0]
    num[col].plot.hist(bins=50, figsize=(8, 4), title=col)
    plt.tight_layout()
else:
    print('No hay columnas numéricas directas: explora df por tu cuenta.')


### ¿Y si el fichero es más grande que la memoria?

Tu sesión tiene un límite de **4 GB de RAM**, pero puedes analizar ficheros de
10 GB o más sin cargarlos enteros:

- **Lee solo las columnas que necesitas**: `pd.read_csv(f, usecols=[...])` /
  `pd.read_parquet(f, columns=[...])`.
- **Procesa por trozos** y acumula solo el resultado:
  ```python
  total = 0
  for chunk in pd.read_csv(fichero, chunksize=1_000_000):
      total += len(chunk)
  ```
- **Consulta con SQL sin cargar nada** — DuckDB (ya instalado) lee CSV y Parquet
  directamente del disco y solo trae a memoria el resultado:
  ```python
  import duckdb
  duckdb.sql("SELECT columna, count(*) FROM '/srv/data/.../fichero.parquet' GROUP BY columna").df()
  ```


## Tu turno

Esto es solo el punto de partida. Algunas ideas:

- Consulta el **reto del dataset** en su [ficha del CSDH](https://data.ibercivis.es/datasets/leuven-urban-weather/).
- **Trabaja sobre una copia**: clic derecho en el fichero → *Duplicate* (o *Save Notebook As…*).
  Tus cambios solo viven en tu espacio del Hub — nunca se suben a GitHub.
- ¿Editaste este notebook y quieres recuperar el original? Usa la celda **Restaurar**
  de aquí abajo (o el notebook [`restaurar.ipynb`](restaurar.ipynb)).
- Dudas y resultados: en el [foro de la plataforma](https://github.com/Ibercivis/citizen-science-data/discussions).

*Atribución: datos de [Leuven.cool Citizen Science Urban Weather Station Network](https://zenodo.org/records/14893734), licencia CC-BY-NC-4.0. Notebook del
Citizen Science Data Hub (CSDH) — Fundación Ibercivis.*


In [ ]:
# ⚠️ RESTAURAR: esto DESCARTA TUS CAMBIOS en este notebook y lo deja como el original.
# 1. Descomenta la línea de abajo (quita el #)   2. Ejecuta la celda
# 3. Después: menú File → Reload Notebook from Disk

# !git -C ~/citizen-science-data fetch -q origin && git -C ~/citizen-science-data checkout origin/main -- leuven-urban-weather.ipynb && echo "Restaurado. Ahora: File → Reload Notebook from Disk"
